# Stronger Cleaner Experiment: DeepDetect
Standalone notebook testing three improvements to the DegradationAwareCleaner.
No changes to the codebase, everything lives here.
The best approach will be transferred after evaluation.

Baseline to beat: **72.0% test accuracy** (P1 phase channel, v5 patch, no aug, 30 epochs)
Note: we run WITH augmentations here. The whole point of a stronger cleaner
is to handle degradations so augmentations can be turned back on.

| ID | Approach | Description |
|---|---|---|
| C1 | Deeper filters | Replace single Conv2d with 3-layer residual blocks per filter |
| C2 | Deeper perception | 3-layer perception module with BatchNorm |
| C3 | C1 + C2 combined | Full stronger cleaner - both improvements together |

All experiments use v5 patch selector, phase channel (P1), WITH augmentations, 30 epochs.
If any C beats 72.0% with augmentations on, it means the cleaner is successfully
restoring the frequency signal that augmentations destroy.


## 1. Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.amp import GradScaler, autocast
import importlib

print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")
if torch.cuda.is_available():
    print(f"GPU:    {torch.cuda.get_device_name(0)}")
    print(f"VRAM:   {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")


Device: cuda
GPU:    NVIDIA RTX PRO 4000 Blackwell
VRAM:   25.2 GB


## 2. Data: v5 patch, WITH augmentations

In [2]:
from config import Config
from data.deepdetect import get_deepdetect_loaders

def make_cfg(experiment_name, epochs=30):
    cfg = Config()
    cfg.data.deepdetect_root   = "../data/raw/deep_detect/data"
    cfg.data.dataset           = "deepdetect"
    cfg.data.image_size        = 224
    cfg.data.batch_size        = 2048
    cfg.data.num_workers       = 4
    cfg.backbone.img_size      = 224
    cfg.frequency.patch_size   = 56
    # ── Augmentations ON — cleaner must handle these ──
    cfg.data.jpeg_aug          = True
    cfg.data.blur_aug          = True
    cfg.data.noise_aug         = True
    cfg.data.recompression_aug = True
    cfg.data.mixed_aug         = True
    cfg.data.mixed_aug_prob    = 0.3
    cfg.train.epochs           = epochs
    cfg.train.backbone_lr      = cfg.train.lr
    cfg.experiment_name        = experiment_name
    cfg.notes                  = f"cleaner experiment: {experiment_name}"
    return cfg

cfg_data = make_cfg("cleaner_setup")
train_loader, val_loader, test_loader = get_deepdetect_loaders(cfg_data)


Train: 76,848  Val: 13,561  Test: 21,776


## 3. Shared Utilities: v5 patch + phase FFT

In [ ]:
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
IMAGENET_STD  = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)

def denorm(t):
    return (t * IMAGENET_STD + IMAGENET_MEAN).clamp(0, 1)

def rgb_to_hsv(rgb):
    r, g, b = rgb[0], rgb[1], rgb[2]
    maxc = rgb.max(0).values
    minc = rgb.min(0).values
    v    = maxc
    s    = torch.where(maxc != 0, (maxc - minc) / (maxc + 1e-8),
                       torch.zeros_like(maxc))
    rc   = (maxc - r) / (maxc - minc + 1e-8)
    gc   = (maxc - g) / (maxc - minc + 1e-8)
    bc   = (maxc - b) / (maxc - minc + 1e-8)
    h    = torch.where(r == maxc, bc - gc,
           torch.where(g == maxc, 2.0 + rc - bc, 4.0 + gc - rc))
    h    = (h / 6.0) % 1.0
    return h, s, v

def rgb_to_hsv_batch(rgb):
    """Batched HSV conversion. rgb: (B, 3, H, W) → h, s, v each (B, H, W). Stays on device."""
    r, g, b = rgb[:,0], rgb[:,1], rgb[:,2]
    maxc = rgb.max(1).values
    minc = rgb.min(1).values
    v    = maxc
    s    = torch.where(maxc != 0, (maxc - minc) / (maxc + 1e-8),
                       torch.zeros_like(maxc))
    rc   = (maxc - r) / (maxc - minc + 1e-8)
    gc   = (maxc - g) / (maxc - minc + 1e-8)
    bc   = (maxc - b) / (maxc - minc + 1e-8)
    h    = torch.where(r == maxc, bc - gc,
           torch.where(g == maxc, 2.0 + rc - bc, 4.0 + gc - rc))
    h    = (h / 6.0) % 1.0
    return h, s, v


def select_patch_v5_batch(images, patch_size=56):
    B, C, H, W = images.shape
    patch_size = min(patch_size, H, W)
    if patch_size == H and patch_size == W:
        return images

    device = images.device

    # Variance map — all on GPU
    gray   = images.mean(dim=1, keepdim=True)  # (B, 1, H, W)
    kernel = torch.ones(1, 1, patch_size, patch_size,
                        device=device) / (patch_size ** 2)
    local_mean    = F.conv2d(gray,      kernel, padding=0)  # (B, 1, H', W')
    local_mean_sq = F.conv2d(gray ** 2, kernel, padding=0)
    local_var     = (local_mean_sq - local_mean ** 2).squeeze(1)  # (B, H', W')

    # Skin density map — all on GPU, no .cpu() transfer
    mean_gpu = IMAGENET_MEAN.to(device)
    std_gpu  = IMAGENET_STD.to(device)
    img_denorm = (images * std_gpu + mean_gpu).clamp(0, 1)
    h, s, v    = rgb_to_hsv_batch(img_denorm)
    skin_mask  = ((h >= 0.0) & (h <= 0.1) & (s >= 0.1) &
                  (s <= 0.7) & (v >= 0.2)).float().unsqueeze(1)  # (B, 1, H, W)
    skin_kernel = torch.ones(1, 1, patch_size, patch_size,
                             device=device) / (patch_size ** 2)
    local_skin  = F.conv2d(skin_mask, skin_kernel, padding=0).squeeze(1)  # (B, H', W')

    # Per-image argmax and slice — only B=64 iterations, fast
    patches = []
    for i in range(B):
        skin_present = (local_skin[i] >= 0.3)
        if skin_present.any():
            var_masked = local_var[i].clone()
            var_masked[~skin_present] = float("inf")
            flat_idx = var_masked.argmin()
        else:
            flat_idx = local_var[i].argmin()
        top  = (flat_idx // local_var.shape[2]).item()
        left = (flat_idx  % local_var.shape[2]).item()
        patches.append(images[i, :, top:top+patch_size, left:left+patch_size])

    return torch.stack(patches)

def phase_fft(x, fftshift=True):
    """Log-magnitude + normalised phase concatenated. (B, 2C, H, W)"""
    spectrum = torch.fft.fft2(x.float())
    if fftshift:
        spectrum = torch.fft.fftshift(spectrum, dim=(-2,-1))
    log_mag = torch.log1p(torch.abs(spectrum))
    mean = log_mag.mean(dim=(-2,-1), keepdim=True)
    std  = log_mag.std(dim=(-2,-1), keepdim=True)
    log_mag = (log_mag - mean) / (std + 1e-8)
    phase = torch.angle(spectrum) / torch.pi
    return torch.cat([log_mag, phase], dim=1)

from models.frequency_branch import SRMFilterLayer

from utils.metrics import binary_accuracy
from pathlib import Path

def train_and_eval(model, cfg, train_loader, val_loader, test_loader, device):
    epochs    = cfg.train.epochs
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
    scaler    = GradScaler(enabled=device.type == "cuda")
    Path(cfg.train.checkpoint_dir).mkdir(parents=True, exist_ok=True)

    print(f"\n{'='*65}")
    print(f"Experiment: {cfg.experiment_name} | Epochs: {epochs}")
    print(f"{'='*65}")

    best_val_acc = 0.0

    for epoch in range(epochs):
        model.train()
        total_loss, train_logits, train_labels = 0.0, [], []

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            with autocast(device_type=device.type, enabled=device.type == "cuda"):
                logits = model(images)
                loss   = criterion(logits, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            total_loss += loss.item()
            train_logits.append(logits.detach().cpu())
            train_labels.append(labels.cpu())

        scheduler.step()
        train_acc = binary_accuracy(torch.cat(train_logits), torch.cat(train_labels))

        model.eval()
        val_logits, val_labels = [], []
        with torch.no_grad():
            for images, labels in val_loader:
                val_logits.append(model(images.to(device)).cpu())
                val_labels.append(labels)
        val_acc = binary_accuracy(torch.cat(val_logits), torch.cat(val_labels))

        print(f"Epoch {epoch+1:>3}/{epochs} | "
              f"train_loss={total_loss/len(train_loader):.4f} | "
              f"train_acc={train_acc:.1%} | val_acc={val_acc:.1%}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(),
                       f"{cfg.train.checkpoint_dir}/best_{cfg.experiment_name}.pt")
            print(f"  -> Saved best val_acc={best_val_acc:.1%}")

    model.load_state_dict(torch.load(
        f"{cfg.train.checkpoint_dir}/best_{cfg.experiment_name}.pt",
        map_location=device))
    model.eval()
    all_logits, all_labels = [], []
    with torch.no_grad():
        for images, labels in test_loader:
            all_logits.append(model(images.to(device)).cpu())
            all_labels.append(labels)
    acc = binary_accuracy(torch.cat(all_logits), torch.cat(all_labels))
    print(f"\nTest accuracy: {acc:.1%}")
    return acc

## 4. Cleaner Variants
Three cleaner designs tested here. All are drop-in replacements for
`DegradationAwareCleaner` — same interface, stronger architecture.


In [ ]:
class ResidualBlock(nn.Module):
    """
    3-layer residual block: Conv → BN → ReLU → Conv → BN → ReLU → Conv → BN
    with skip connection. Much more capacity than a single Conv2d.
    """
    def __init__(self, channels=3):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(channels, 16, 3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),
            nn.Conv2d(16, 16, 3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),
            nn.Conv2d(16, channels, 3, padding=1),
            nn.BatchNorm2d(channels),
        )

    def forward(self, x):
        return x + self.block(x)  

class CleanerC1(nn.Module):
    """Replace single Conv2d filters with residual blocks. Perception unchanged."""
    def __init__(self):
        super().__init__()
        self.perception = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(True), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(True),
            nn.AdaptiveAvgPool2d(1),
        )
        self.perc_head = nn.Linear(32, 3)

        # Deeper filters — residual blocks instead of single Conv2d
        self.filter_clean      = ResidualBlock(3)
        self.filter_blurry     = ResidualBlock(3)
        self.filter_compressed = ResidualBlock(3)

    def forward(self, x):
        w = F.softmax(self.perc_head(
            self.perception(x).flatten(1)), dim=1)  # (B, 3)
        out_c = self.filter_clean(x)
        out_b = self.filter_blurry(x)
        out_k = self.filter_compressed(x)
        blended = (w[:,0:1].unsqueeze(-1).unsqueeze(-1) * out_c +
                   w[:,1:2].unsqueeze(-1).unsqueeze(-1) * out_b +
                   w[:,2:3].unsqueeze(-1).unsqueeze(-1) * out_k)
        return x + blended

    def reconstruction_loss(self, x):
        return F.l1_loss(self.forward(x), x)

class CleanerC2(nn.Module):
    """Deeper perception module with BatchNorm. Filters unchanged (single Conv2d)."""
    def __init__(self):
        super().__init__()
        self.perception = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU(True), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(True), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(True),
            nn.AdaptiveAvgPool2d(1),
        )
        self.perc_head = nn.Linear(64, 3)
        self.filter_clean      = nn.Conv2d(3, 3, 3, padding=1)
        self.filter_blurry     = nn.Conv2d(3, 3, 3, padding=1)
        self.filter_compressed = nn.Conv2d(3, 3, 3, padding=1)

    def forward(self, x):
        w = F.softmax(self.perc_head(
            self.perception(x).flatten(1)), dim=1)
        out_c = self.filter_clean(x)
        out_b = self.filter_blurry(x)
        out_k = self.filter_compressed(x)
        blended = (w[:,0:1].unsqueeze(-1).unsqueeze(-1) * out_c +
                   w[:,1:2].unsqueeze(-1).unsqueeze(-1) * out_b +
                   w[:,2:3].unsqueeze(-1).unsqueeze(-1) * out_k)
        return x + blended

    def reconstruction_loss(self, x):
        return F.l1_loss(self.forward(x), x)
    
class CleanerC3(nn.Module):
    """Deeper perception (with BN) + residual block filters. Full stronger cleaner."""
    def __init__(self):
        super().__init__()
        self.perception = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU(True), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(True), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(True),
            nn.AdaptiveAvgPool2d(1),
        )
        self.perc_head = nn.Linear(64, 3)
        self.filter_clean      = ResidualBlock(3)
        self.filter_blurry     = ResidualBlock(3)
        self.filter_compressed = ResidualBlock(3)

    def forward(self, x):
        w = F.softmax(self.perc_head(
            self.perception(x).flatten(1)), dim=1)
        out_c = self.filter_clean(x)
        out_b = self.filter_blurry(x)
        out_k = self.filter_compressed(x)
        blended = (w[:,0:1].unsqueeze(-1).unsqueeze(-1) * out_c +
                   w[:,1:2].unsqueeze(-1).unsqueeze(-1) * out_b +
                   w[:,2:3].unsqueeze(-1).unsqueeze(-1) * out_k)
        return x + blended

    def reconstruction_loss(self, x):
        return F.l1_loss(self.forward(x), x)

## 5. Full Frequency Branch with Swappable Cleaner

In [ ]:
class FreqBranchWithCleaner(nn.Module):
    """
    Full frequency pipeline with a swappable cleaner.
    Uses v5 patch selector + phase FFT (P1) + SRM + custom CNN.
    Cleaner is passed as an argument — swap C1/C2/C3 without changing other code.
    """
    def __init__(self, cleaner, feature_dim=256):
        super().__init__()
        self.cleaner = cleaner
        self.srm     = SRMFilterLayer()
        # SRM: 9ch, phase doubles to 18ch
        self.cnn = nn.Sequential(
            nn.Conv2d(18, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(True), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(True), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(True), nn.MaxPool2d(2),
            nn.Conv2d(128, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(True), nn.MaxPool2d(2),
        )
        self.gap  = nn.AdaptiveAvgPool2d(1)
        self.proj = nn.Linear(128, feature_dim)
        self.head = nn.Linear(feature_dim, 2)

    def forward(self, images):
        patches = select_patch_v5_batch(images, patch_size=56).to(images.device)
        patches = self.cleaner(patches)          # clean the patch
        x = self.srm(patches)                    # SRM noise residual
        x = phase_fft(x)                         # log-mag + phase (18ch)
        x = self.cnn(x)
        x = self.gap(x).flatten(1)
        x = self.proj(x)
        return self.head(x)

## 6. Run All Three Cleaner Variants

In [6]:
results_cleaner = {}
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [7]:
# ── C1: Deeper filters ────────────────────────────────────────────────────────
cfg_c1  = make_cfg("cleaner_c1_deep_filters")
model_c1 = FreqBranchWithCleaner(CleanerC1()).to(device)
results_cleaner["C1 — deeper filters"]  = train_and_eval(
    model_c1, cfg_c1, train_loader, val_loader, test_loader, device)



Experiment: cleaner_c1_deep_filters | Epochs: 30
Epoch   1/30 | train_loss=0.7113 | train_acc=53.0% | val_acc=54.2%
  -> Saved best val_acc=54.2%
Epoch   2/30 | train_loss=0.6836 | train_acc=56.1% | val_acc=55.7%
  -> Saved best val_acc=55.7%
Epoch   3/30 | train_loss=0.6795 | train_acc=57.1% | val_acc=54.5%
Epoch   4/30 | train_loss=0.6774 | train_acc=57.4% | val_acc=55.1%
Epoch   5/30 | train_loss=0.6766 | train_acc=57.5% | val_acc=56.9%
  -> Saved best val_acc=56.9%
Epoch   6/30 | train_loss=0.6724 | train_acc=58.1% | val_acc=56.1%
Epoch   7/30 | train_loss=0.6700 | train_acc=58.5% | val_acc=47.3%
Epoch   8/30 | train_loss=0.6695 | train_acc=58.8% | val_acc=51.6%
Epoch   9/30 | train_loss=0.6656 | train_acc=59.2% | val_acc=56.6%
Epoch  10/30 | train_loss=0.6643 | train_acc=59.3% | val_acc=57.9%
  -> Saved best val_acc=57.9%
Epoch  11/30 | train_loss=0.6619 | train_acc=59.9% | val_acc=58.1%
  -> Saved best val_acc=58.1%
Epoch  12/30 | train_loss=0.6626 | train_acc=59.9% | val_acc=57

In [8]:
# ── C2: Deeper perception ─────────────────────────────────────────────────────
cfg_c2  = make_cfg("cleaner_c2_deep_perception")
model_c2 = FreqBranchWithCleaner(CleanerC2()).to(device)
results_cleaner["C2 — deeper perception"] = train_and_eval(
    model_c2, cfg_c2, train_loader, val_loader, test_loader, device)



Experiment: cleaner_c2_deep_perception | Epochs: 30
Epoch   1/30 | train_loss=0.7181 | train_acc=52.3% | val_acc=54.0%
  -> Saved best val_acc=54.0%
Epoch   2/30 | train_loss=0.6842 | train_acc=55.5% | val_acc=54.5%
  -> Saved best val_acc=54.5%
Epoch   3/30 | train_loss=0.6792 | train_acc=57.0% | val_acc=55.5%
  -> Saved best val_acc=55.5%
Epoch   4/30 | train_loss=0.6739 | train_acc=57.9% | val_acc=54.3%
Epoch   5/30 | train_loss=0.6725 | train_acc=58.4% | val_acc=47.5%
Epoch   6/30 | train_loss=0.6689 | train_acc=58.7% | val_acc=54.3%
Epoch   7/30 | train_loss=0.6690 | train_acc=58.8% | val_acc=48.3%
Epoch   8/30 | train_loss=0.6663 | train_acc=59.2% | val_acc=58.6%
  -> Saved best val_acc=58.6%
Epoch   9/30 | train_loss=0.6648 | train_acc=59.4% | val_acc=56.0%
Epoch  10/30 | train_loss=0.6621 | train_acc=59.9% | val_acc=57.2%
Epoch  11/30 | train_loss=0.6612 | train_acc=60.1% | val_acc=57.1%
Epoch  12/30 | train_loss=0.6598 | train_acc=60.4% | val_acc=57.6%
Epoch  13/30 | train_lo

In [9]:
# ── C3: Combined ──────────────────────────────────────────────────────────────
cfg_c3  = make_cfg("cleaner_c3_combined")
model_c3 = FreqBranchWithCleaner(CleanerC3()).to(device)
results_cleaner["C3 — combined (C1+C2)"] = train_and_eval(
    model_c3, cfg_c3, train_loader, val_loader, test_loader, device)



Experiment: cleaner_c3_combined | Epochs: 30
Epoch   1/30 | train_loss=0.7271 | train_acc=53.0% | val_acc=55.7%
  -> Saved best val_acc=55.7%
Epoch   2/30 | train_loss=0.6846 | train_acc=55.7% | val_acc=56.2%
  -> Saved best val_acc=56.2%
Epoch   3/30 | train_loss=0.6788 | train_acc=57.3% | val_acc=49.8%
Epoch   4/30 | train_loss=0.6761 | train_acc=57.6% | val_acc=56.4%
  -> Saved best val_acc=56.4%
Epoch   5/30 | train_loss=0.6718 | train_acc=58.3% | val_acc=48.7%
Epoch   6/30 | train_loss=0.6705 | train_acc=58.3% | val_acc=56.3%
Epoch   7/30 | train_loss=0.6686 | train_acc=58.8% | val_acc=52.1%
Epoch   8/30 | train_loss=0.6634 | train_acc=59.6% | val_acc=58.3%
  -> Saved best val_acc=58.3%
Epoch   9/30 | train_loss=0.6626 | train_acc=59.9% | val_acc=56.9%
Epoch  10/30 | train_loss=0.6616 | train_acc=60.1% | val_acc=56.3%
Epoch  11/30 | train_loss=0.6609 | train_acc=60.1% | val_acc=58.3%
  -> Saved best val_acc=58.3%
Epoch  12/30 | train_loss=0.6572 | train_acc=60.4% | val_acc=58.6%


In [ ]:
import torchvision.models as tvm

class PerceptualLoss(nn.Module):
    """
    VGG16 feature matching loss.
    Extracts features at relu2_2 and relu3_3 — mid-level features that
    capture texture and structure without being too low (pixels) or too
    high (semantics).
    MAE + λ * perceptual for stable training.
    """
    def __init__(self, lambda_perc=0.1):
        super().__init__()
        self.lambda_perc = lambda_perc
        vgg = tvm.vgg16(weights=tvm.VGG16_Weights.DEFAULT).features
        # relu2_2 = layer 9, relu3_3 = layer 16
        self.slice1 = nn.Sequential(*list(vgg.children())[:10]).eval()
        self.slice2 = nn.Sequential(*list(vgg.children())[10:17]).eval()
        for p in self.parameters():
            p.requires_grad = False

    def forward(self, cleaned, original):
        # MAE pixel loss
        mae = F.l1_loss(cleaned, original)

        # Perceptual loss at two VGG layers
        f1_c = self.slice1(cleaned)
        f1_o = self.slice1(original)
        f2_c = self.slice2(f1_c)
        f2_o = self.slice2(f1_o)
        perc = F.l1_loss(f1_c, f1_o) + F.l1_loss(f2_c, f2_o)

        return mae + self.lambda_perc * perc

class CleanerC4(nn.Module):
    """C3 architecture (deeper perception + residual filters) trained with
    MAE + perceptual loss instead of MAE alone."""
    def __init__(self):
        super().__init__()
        self.perception = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU(True), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(True), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(True),
            nn.AdaptiveAvgPool2d(1),
        )
        self.perc_head = nn.Linear(64, 3)
        self.filter_clean      = ResidualBlock(3)
        self.filter_blurry     = ResidualBlock(3)
        self.filter_compressed = ResidualBlock(3)
        self.loss_fn = PerceptualLoss(lambda_perc=0.1)

    def forward(self, x):
        w = F.softmax(self.perc_head(
            self.perception(x).flatten(1)), dim=1)
        out_c = self.filter_clean(x)
        out_b = self.filter_blurry(x)
        out_k = self.filter_compressed(x)
        blended = (w[:,0:1].unsqueeze(-1).unsqueeze(-1) * out_c +
                   w[:,1:2].unsqueeze(-1).unsqueeze(-1) * out_b +
                   w[:,2:3].unsqueeze(-1).unsqueeze(-1) * out_k)
        return x + blended

    def reconstruction_loss(self, x):
        cleaned = self.forward(x)
        return self.loss_fn(cleaned, x)

In [ ]:
cfg_c4   = make_cfg("cleaner_c4_perceptual")
model_c4 = FreqBranchWithCleaner(CleanerC4()).to(device)
results_cleaner["C4 — C3 + perceptual loss"] = train_and_eval(
    model_c4, cfg_c4, train_loader, val_loader, test_loader, device)

Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /home/peter-kabati/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [01:52<00:00, 4.92MB/s] 



Experiment: cleaner_c4_perceptual | Epochs: 30
Epoch   1/30 | train_loss=0.7097 | train_acc=53.2% | val_acc=54.2%
  -> Saved best val_acc=54.2%
Epoch   2/30 | train_loss=0.6848 | train_acc=55.3% | val_acc=54.3%
  -> Saved best val_acc=54.3%
Epoch   3/30 | train_loss=0.6813 | train_acc=56.5% | val_acc=54.2%
Epoch   4/30 | train_loss=0.6794 | train_acc=56.9% | val_acc=56.9%
  -> Saved best val_acc=56.9%
Epoch   5/30 | train_loss=0.6738 | train_acc=57.9% | val_acc=52.6%
Epoch   6/30 | train_loss=0.6726 | train_acc=58.4% | val_acc=54.1%
Epoch   7/30 | train_loss=0.6695 | train_acc=58.8% | val_acc=53.9%
Epoch   8/30 | train_loss=0.6665 | train_acc=59.3% | val_acc=54.2%
Epoch   9/30 | train_loss=0.6641 | train_acc=59.7% | val_acc=56.1%
Epoch  10/30 | train_loss=0.6620 | train_acc=59.9% | val_acc=57.0%
  -> Saved best val_acc=57.0%
Epoch  11/30 | train_loss=0.6569 | train_acc=60.7% | val_acc=48.0%
Epoch  12/30 | train_loss=0.6580 | train_acc=60.6% | val_acc=55.3%
Epoch  13/30 | train_loss=0.

## 7. Results

In [ ]:
baseline_no_aug  = 0.712   # v5, no aug, 30 epochs
baseline_phase   = 0.720   # P1 phase, no aug, 30 epochs
baseline_with_aug = 0.632  # v5, with aug, 30 epochs — what cleaner needs to beat

print("\n" + "="*65)
print("CLEANER EXPERIMENT — DeepDetect, 30 epochs, WITH augmentations")
print("="*65)
print(f"  {'Reference — v5, no aug':<40} {baseline_no_aug:.1%}")
print(f"  {'Reference — P1 phase, no aug':<40} {baseline_phase:.1%}")
print(f"  {'Reference — v5, with aug (no cleaner)':<40} {baseline_with_aug:.1%}")
print("-"*65)

for name, acc in results_cleaner.items():
    delta  = acc - baseline_with_aug
    status = "PASS" if acc >= 0.70 else ("WEAK" if acc >= 0.60 else "FAIL ✗")
    print(f"  {name:<40} {acc:.1%}  ({delta:+.1%})  {status}")

print("="*65)

best_name = max(results_cleaner, key=results_cleaner.get)
best_acc  = results_cleaner[best_name]
print(f"\nBest cleaner: {best_name}  ({best_acc:.1%})")

if best_acc >= baseline_phase:
    print("SUCCESS - stronger cleaner beats no-aug baseline even with augmentations on.")
    print("Transfer winning cleaner to codebase.")
elif best_acc >= baseline_with_aug:
    print("IMPROVEMENT - cleaner helps vs no cleaner with aug, but below no-aug ceiling.")
    print("Partial success - still worth transferring if improvement is meaningful.")
else:
    print("NO IMPROVEMENT - cleaner does not help with augmentations.")
    print("Proceed with no-aug training and document as a limitation.")



CLEANER EXPERIMENT — DeepDetect, 30 epochs, WITH augmentations
  Reference — v5, no aug                   71.2%
  Reference — P1 phase, no aug             72.0%
  Reference — v5, with aug (no cleaner)    63.2%
-----------------------------------------------------------------
  C1 — deeper filters                      64.6%  (+1.4%)  WEAK
  C2 — deeper perception                   64.0%  (+0.8%)  WEAK
  C3 — combined (C1+C2)                    63.6%  (+0.4%)  WEAK

Best cleaner: C1 — deeper filters  (64.6%)
IMPROVEMENT — cleaner helps vs no cleaner with aug, but below no-aug ceiling.
Partial success — still worth transferring if improvement is meaningful.
